# Phase 1: Dataset Discovery & EDF Metadata/Channel Audit

This notebook orchestrates the `imagined_speech` package on Google Colab or locally to:
1. Discover recordings across `Char`, `Digit`, and `Image` directories.
2. Parse and normalize labels while preserving raw filenames, relative paths, and marking any extracted prefixes as `unverified_candidate_prefix` (Rules 11 & 12).
3. Inspect binary EDF headers and in-memory signal properties to distinguish true EEG channels from non-EEG metadata/counter/sensor channels (Rule 10).

In [ ]:
# Optional: Mount Google Drive when running on Google Colab
import os
from pathlib import Path

try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f'Running in Google Colab: {IN_COLAB}')

In [ ]:
# Configure dataset root and output paths via environment variables (no hard-coded machine paths)
DATA_ROOT = Path(os.environ.get('ISR_DATA_ROOT', './data/raw'))
OUTPUT_DIR = Path(os.environ.get('ISR_OUTPUT_DIR', './outputs/phase1_audit'))
CONFIG_PATH = Path('configs/default_config.yaml')

print(f'DATA_ROOT:  {DATA_ROOT}')
print(f'OUTPUT_DIR: {OUTPUT_DIR}')

In [ ]:
import json
import pandas as pd
from imagined_speech.config import load_config
from imagined_speech.data.edf_audit import run_dataset_audit
from imagined_speech.utils import set_global_seed, setup_logging

config = load_config(
    config_path=CONFIG_PATH if CONFIG_PATH.exists() else None,
    data_root=DATA_ROOT,
    output_dir=OUTPUT_DIR,
)
setup_logging(config.log_level)
set_global_seed(config.seed)

audit_result = run_dataset_audit(config, save_outputs=True)
print(json.dumps(audit_result.summary, indent=2))

In [ ]:
# Inspect generated manifest and channel audit tables
manifest_df = pd.read_csv(OUTPUT_DIR / 'manifest.csv')
channel_df = pd.read_csv(OUTPUT_DIR / 'channel_audit.csv')

display(manifest_df.head(10))
display(channel_df.groupby(['task_category', 'audited_channel_role', 'raw_channel_name']).size().reset_index(name='count'))